In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# -------------------------------
# Step 1: Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------
# Step 2: Load data
# -------------------------------
print("🔹 Loading data...")
data = torch.load("./chemBERTa_dataset.pt", weights_only=False)
print("✅ Data loaded")

# -------------------------------
# Step 3: Prepare tensors (FIXED warning)
# -------------------------------
X = data["X"].clone().detach().float()
y = data["y"].clone().detach().float()

# -------------------------------
# Step 4: Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X.numpy(), y.numpy(), test_size=0.2, random_state=42
)

# Convert back to tensors
X_train = torch.tensor(X_train).to(device)
y_train = torch.tensor(y_train).to(device)
X_test = torch.tensor(X_test).to(device)
y_test = torch.tensor(y_test).to(device)

# -------------------------------
# Step 5: Model (improved)
# -------------------------------
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]

model = nn.Sequential(
    nn.Linear(input_dim, 1024),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, output_dim)
).to(device)

# -------------------------------
# Step 6: Weighted Loss (IMPORTANT)
# -------------------------------
print("🔹 Computing class weights...")

pos_counts = y_train.sum(dim=0)
neg_counts = y_train.shape[0] - pos_counts
pos_weight = (neg_counts / (pos_counts + 1e-5)).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -------------------------------
# Step 7: Optimizer
# -------------------------------
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------------
# Step 8: Training
# -------------------------------
epochs = 20
batch_size = 256

print("🔹 Training...")

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train.size(0))

    epoch_loss = 0

    for i in tqdm(range(0, X_train.size(0), batch_size), desc=f"Epoch {epoch+1}"):
        idx = perm[i:i+batch_size]
        batch_X = X_train[idx]
        batch_y = y_train[idx]

        optimizer.zero_grad()
        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

print("✅ Training complete")

# -------------------------------
# Step 9: Prediction
# -------------------------------
print("🔹 Predicting...")
model.eval()

with torch.no_grad():
    logits = model(X_test)
    probs = torch.sigmoid(logits)

    # 🔥 Improved threshold
    y_pred = (probs > 0.5).float()

# -------------------------------
# Step 10: Evaluation
# -------------------------------
y_pred_np = y_pred.cpu().numpy()
y_test_np = y_test.cpu().numpy()

score = f1_score(y_test_np, y_pred_np, average="micro")
print("🎯 F1 Score:", score)

Using device: cuda
🔹 Loading data...
✅ Data loaded
🔹 Computing class weights...
🔹 Training...


Epoch 1: 100%|██████████| 198/198 [00:01<00:00, 128.25it/s]


Epoch 1 Loss: 241.1578


Epoch 2: 100%|██████████| 198/198 [00:01<00:00, 129.65it/s]


Epoch 2 Loss: 224.1580


...

Epoch 18 Loss: 173.2643


Epoch 19: 100%|██████████| 198/198 [00:01<00:00, 130.60it/s]


Epoch 19 Loss: 172.3991


Epoch 20: 100%|██████████| 198/198 [00:01<00:00, 126.08it/s]


Epoch 20 Loss: 171.1619
✅ Training complete
🔹 Predicting...
🎯 F1 Score: 0.30428852938832585


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------
# Step 2: Load data
# -------------------------------
print("Loading data...")
data = torch.load("./chemBERTa_dataset.pt", weights_only=False)
print("Data loaded")

# -------------------------------
# Step 3: Prepare tensors (FIXED warning)
# -------------------------------
X = data["X"].clone().detach().float()
y = data["y"].clone().detach().float()

# -------------------------------
# Step 4: Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X.numpy(), y.numpy(), test_size=0.2, random_state=42
)

# Convert back to tensors
X_train = torch.tensor(X_train).to(device)
y_train = torch.tensor(y_train).to(device)
X_test = torch.tensor(X_test).to(device)
y_test = torch.tensor(y_test).to(device)

# -------------------------------
# Step 5: Model (improved)
# -------------------------------
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]

model = nn.Sequential(
    nn.Linear(input_dim, 1024),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, output_dim)
).to(device)

# -------------------------------
# Step 6: Weighted Loss (IMPORTANT)
# -------------------------------
print("🔹 Computing class weights...")

pos_counts = y_train.sum(dim=0)
neg_counts = y_train.shape[0] - pos_counts
pos_weight = (neg_counts / (pos_counts + 1e-5)).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -------------------------------
# Step 7: Optimizer
# -------------------------------
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------------
# Step 8: Training
# -------------------------------
epochs = 20
batch_size = 256

print("🔹 Training...")

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train.size(0))

    epoch_loss = 0

    for i in tqdm(range(0, X_train.size(0), batch_size), desc=f"Epoch {epoch+1}"):
        idx = perm[i:i+batch_size]
        batch_X = X_train[idx]
        batch_y = y_train[idx]

        optimizer.zero_grad()
        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

print("✅ Training complete")

# -------------------------------
# Step 9: Prediction
# -------------------------------
print("🔹 Predicting...")
model.eval()

with torch.no_grad():
    logits = model(X_test)
    probs = torch.sigmoid(logits)

    # 🔥 Improved threshold
    y_pred = (probs > 0.5).float()

# -------------------------------
# Step 10: Evaluation
# -------------------------------
from sklearn.metrics import f1_score, roc_auc_score

print("🔹 Predicting...")
model.eval()

with torch.no_grad():
    logits = model(X_test)
    probs = torch.sigmoid(logits)   # 🔥 probabilities (IMPORTANT)

    # F1 prediction (threshold-based)
    y_pred = (probs > 0.5).float()

# Convert to numpy
y_pred_np = y_pred.cpu().numpy()
y_test_np = y_test.cpu().numpy()
probs_np = probs.cpu().numpy()

# -------------------------------
# F1 Score
# -------------------------------
f1 = f1_score(y_test_np, y_pred_np, average="micro")
print("🎯 F1 Score:", f1)

# -------------------------------
# AUC Score (IMPORTANT)
# -------------------------------
try:
    auc = roc_auc_score(y_test_np, probs_np, average="micro")
    print("📈 ROC-AUC Score:", auc)
except Exception as e:
    print("AUC could not be computed:", e)

Using device: cuda
Loading data...
Data loaded
🔹 Computing class weights...
🔹 Training...


Epoch 1: 100%|██████████| 198/198 [00:01<00:00, 104.78it/s]


Epoch 1 Loss: 241.3728


Epoch 2: 100%|██████████| 198/198 [00:01<00:00, 100.43it/s]


Epoch 2 Loss: 224.6221


...

Epoch 18 Loss: 173.4740


Epoch 19: 100%|██████████| 198/198 [00:01<00:00, 119.22it/s]


Epoch 19 Loss: 172.3013


Epoch 20: 100%|██████████| 198/198 [00:01<00:00, 121.99it/s]


Epoch 20 Loss: 171.2326
✅ Training complete
🔹 Predicting...
🔹 Predicting...
🎯 F1 Score: 0.30215632744263476
📈 ROC-AUC Score: 0.8195151512157498
